In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.patches as mpatches

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w'),  # Write to file
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
COV_PATH = f'/confess/dicarlo/02-confess/05-confess-post-data/'
LAI_PATH = f'/confess/dicarlo/00-dataset/02-confess/01-confess_data/'
SAVE_PATH = f'/confess/dicarlo/cartella_ordinata/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
dset_ctrl = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

dset_ctrl_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_ctrl_em_ocean= xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_em_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

ld = xr.DataArray(leads, dims='lead_time', name='lead_time')
dset_ctrl['lead_time']=ld
dset_sens['lead_time']=ld
dset_ctrl_em['lead_time']=ld
dset_sens_em['lead_time']=ld

dset_ctrl_land['lead_time']=ld
dset_sens_land['lead_time']=ld
dset_ctrl_em_land['lead_time']=ld
dset_sens_em_land['lead_time']=ld


dset_ctrl_ocean['lead_time']=ld
dset_sens_ocean['lead_time']=ld
dset_ctrl_em_ocean['lead_time']=ld
dset_sens_em_ocean['lead_time']=ld

# Convert the dataset to a DataFrame
dset_ctrl = dset_ctrl.to_dataframe().reset_index()
dset_sens = dset_sens.to_dataframe().reset_index()
dset_ctrl_em = dset_ctrl_em.to_dataframe().reset_index()
dset_sens_em = dset_sens_em.to_dataframe().reset_index()

dset_ctrl_land = dset_ctrl_land.to_dataframe().reset_index()
dset_sens_land = dset_sens_land.to_dataframe().reset_index()
dset_ctrl_em_land = dset_ctrl_em_land.to_dataframe().reset_index()
dset_sens_em_land = dset_sens_em_land.to_dataframe().reset_index()

dset_ctrl_ocean = dset_ctrl_ocean.to_dataframe().reset_index()
dset_sens_ocean = dset_sens_ocean.to_dataframe().reset_index()
dset_ctrl_em_ocean = dset_ctrl_em_ocean.to_dataframe().reset_index()
dset_sens_em_ocean = dset_sens_em_ocean.to_dataframe().reset_index()

In [ ]:
fig,ax=plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', data=dset_sens, ax=ax, color='lightcoral', linecolor='r')
red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
sns.boxplot(x='lead_time', y='tas', data=dset_ctrl, ax=ax, color='lavender', linecolor='b')
blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')
plt.legend(handles=[red_patch, blue_patch],loc='upper right')

dset_ctrl_em[var].plot(ax=ax,color='k', linestyle='--')
dset_sens_em[var].plot(ax=ax,color='k')
ax.set_title('a) GLOBAL')

fig.savefig(f'/confess/dicarlo/figure-decadale/{var}_acc_global_1999.png', dpi=600, bbox_inches = 'tight')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', data=dset_sens_land, ax=ax, color='lightcoral', linecolor='r')
red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
sns.boxplot(x='lead_time', y='tas', data=dset_ctrl_land, ax=ax, color='lavender', linecolor='b')
blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')

# Force the legend to the upper-right corner
plt.legend(handles=[red_patch, blue_patch], loc='upper right', bbox_to_anchor=(1, 1))

dset_ctrl_em_land[var].plot(ax=ax, color='k', linestyle='--')
dset_sens_em_land[var].plot(ax=ax, color='k')
ax.set_title('b) LAND ONLY')

fig.savefig(f'/confess/dicarlo/figure-decadale/{var}_acc_land_1999.png', dpi=600, bbox_inches='tight')

In [ ]:
fig,ax=plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', data=dset_sens_ocean, ax=ax, color='lightcoral', linecolor='r')
red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
sns.boxplot(x='lead_time', y='tas', data=dset_ctrl_ocean, ax=ax, color='lavender', linecolor='b')
blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')
plt.legend(handles=[red_patch, blue_patch],loc='upper right')

dset_ctrl_em_ocean[var].plot(ax=ax,color='k', linestyle='--')
dset_sens_em_ocean[var].plot(ax=ax,color='k')
ax.set_title('c) OCEAN ONLY')

fig.savefig(f'/confess/dicarlo/figure-decadale/{var}_acc_ocean_1999.png', dpi=600, bbox_inches = 'tight')